# Deep Learning Project — CIFAR-10 Image Classification



## 1. Imports and configuration


In [ ]:

import os
os.environ["PYTHONHASHSEED"] = "0"


In [ ]:
import os
import random as python_random
import pickle

import numpy as np
from matplotlib import pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
python_random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

BATCH_SIZE = 128
EPOCHS = 30
LEARNING_RATE = 1e-3
NUM_CLASSES = 10
IMG_SIZE = 32
TL_IMG_SIZE = 96
VAL_SIZE = 5000


DATA_DIR = "/project/cifar-10-batches-py"

OUT_DIR = "./outputs_tf"
os.makedirs(OUT_DIR, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices("GPU")) > 0)


## 2. Loading CIFAR-10 from Python batch files




In [ ]:
def unpickle(file_path):
    """Load a CIFAR-10 Python batch file."""
    with open(file_path, "rb") as fo:
        data_dict = pickle.load(fo, encoding="latin1")
    return data_dict


def reshape_cifar_batch(data):
    """Convert CIFAR-10 flat vectors into RGB images."""
    images = data.reshape(-1, 3, 32, 32)
    images = images.transpose(0, 2, 3, 1)
    return images.astype("float32")


def load_cifar10_from_py(data_dir=DATA_DIR):
    """Load CIFAR-10 from local Python batch files."""
    x_batches, y_batches = [], []

    for i in range(1, 6):
        batch_path = os.path.join(data_dir, f"data_batch_{i}")
        batch = unpickle(batch_path)

        x_batches.append(reshape_cifar_batch(batch["data"]))
        y_batches.append(np.array(batch["labels"], dtype="int32"))

    test_batch = unpickle(os.path.join(data_dir, "test_batch"))
    x_test = reshape_cifar_batch(test_batch["data"])
    y_test = np.array(test_batch["labels"], dtype="int32")

    x_train = np.concatenate(x_batches, axis=0)
    y_train = np.concatenate(y_batches, axis=0)

    meta = unpickle(os.path.join(data_dir, "batches.meta"))
    class_names = meta["label_names"]

    return (x_train, y_train), (x_test, y_test), class_names


(x_train_full, y_train_full), (x_test, y_test), class_names = load_cifar10_from_py(DATA_DIR)

print("Loaded CIFAR-10 from local Python batch files")
print("Training full:", x_train_full.shape, y_train_full.shape)
print("Test:", x_test.shape, y_test.shape)
print("Classes:", class_names)


## 3. Train / validation / test split


- 45,000 images for training;
- 5,000 images for validation;
- 10,000 images for final testing.


In [ ]:
rng = np.random.default_rng(SEED)
indices = rng.permutation(len(x_train_full))

val_indices = indices[:VAL_SIZE]
train_indices = indices[VAL_SIZE:]

x_train = x_train_full[train_indices]
y_train = y_train_full[train_indices]
x_val = x_train_full[val_indices]
y_val = y_train_full[val_indices]

print("Train:", x_train.shape, y_train.shape)
print("Validation:", x_val.shape, y_val.shape)
print("Test:", x_test.shape, y_test.shape)


## 4. Dataset exploration


In [ ]:
def show_samples(x, y, class_names, n=9):
    plt.figure(figsize=(8, 8))
    for i in range(n):
        ax = plt.subplot(3, 3, i + 1)
        ax.imshow(x[i].astype("uint8"))
        ax.set_title(class_names[int(y[i])])
        ax.axis("off")

    plt.suptitle("Sample images from CIFAR-10", fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "sample_images.png"), dpi=120, bbox_inches="tight")
    plt.show()


show_samples(x_train, y_train, class_names)


## 5. Preprocessing and pipeline

For the custom CNNs:

- `Rescaling(1./255)` brings pixel values from `[0, 255]` to `[0, 1]`;
- `Normalization` standardizes the input using statistics computed on the training set;
- data augmentation is applied only during training and only when explicitly enabled inside the model.


In [ ]:
normalizer = layers.Normalization(axis=-1, name="normalization")
normalizer.adapt(x_train / 255.0)

augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomTranslation(height_factor=0.10, width_factor=0.10, fill_mode="reflect"),
    layers.RandomContrast(0.10),
], name="data_augmentation")


def make_dataset(x, y, training=False, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices((x, y))

    if training:
        ds = ds.shuffle(buffer_size=len(x), seed=SEED, reshuffle_each_iteration=True)

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = make_dataset(x_train, y_train, training=True)
val_ds = make_dataset(x_val, y_val, training=False)
test_ds = make_dataset(x_test, y_test, training=False)

print(train_ds)


In [ ]:
def preview_augmentation(x, augmentation_layer, n=9):
    base = x[0:1] / 255.0

    plt.figure(figsize=(8, 8))
    for i in range(n):
        augmented = augmentation_layer(base, training=True)[0].numpy()
        ax = plt.subplot(3, 3, i + 1)
        ax.imshow(np.clip(augmented, 0, 1))
        ax.axis("off")

    plt.suptitle("Augmentation preview: same image, 9 random variants", fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "augmentation_preview.png"), dpi=120, bbox_inches="tight")
    plt.show()


preview_augmentation(x_train, augmentation)


## 6. Training and evaluation utilities


In [ ]:
def compile_model(model, learning_rate=LEARNING_RATE):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")]
    )
    return model


def get_callbacks(model_name, patience=6):
    return [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=patience,
            restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            min_lr=1e-6
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(OUT_DIR, f"{model_name}.keras"),
            monitor="val_loss",
            save_best_only=True
        )
    ]


def train_model(model, model_name, train_ds=train_ds, val_ds=val_ds, epochs=EPOCHS,
                learning_rate=LEARNING_RATE, patience=6):
    compile_model(model, learning_rate)
    model.summary()

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=get_callbacks(model_name, patience),
        verbose=1
    )

    return history


def train_model_clip(model, model_name, train_ds=train_ds, val_ds=val_ds,
                     epochs=EPOCHS, learning_rate=5e-4, patience=7):
    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=learning_rate,
            clipnorm=1.0
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    model.summary()

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=get_callbacks(model_name, patience),
        verbose=1
    )

    return history
def plot_history(history, title):
    h = history.history
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(h["loss"], label="train")
    plt.plot(h["val_loss"], label="validation")
    plt.title(f"{title} — Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(h["accuracy"], label="train")
    plt.plot(h["val_accuracy"], label="validation")
    plt.title(f"{title} — Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(alpha=0.3)

    plt.tight_layout()
    filename = f"{title.replace(' ', '_').lower()}_curves.png"
    plt.savefig(os.path.join(OUT_DIR, filename), dpi=120, bbox_inches="tight")
    plt.show()


def plot_history_wide(history, title, loss_ylim=(0.0, 0.9), acc_ylim=(0.70, 1.00)):
    """Re-plot an existing training history with wider y-axis limits."""
    h = history.history
    epochs_range = range(len(h["loss"]))

    plt.figure(figsize=(14, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, h["loss"], label="train")
    plt.plot(epochs_range, h["val_loss"], label="validation")
    plt.title(f"{title} — Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.ylim(*loss_ylim)
    plt.grid(alpha=0.3)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, h["accuracy"], label="train")
    plt.plot(epochs_range, h["val_accuracy"], label="validation")
    plt.title(f"{title} — Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.ylim(*acc_ylim)
    plt.grid(alpha=0.3)
    plt.legend()

    plt.tight_layout()
    plt.show()


def evaluate_model(model, dataset, y_true=None, model_name="model"):
    test_loss, test_acc = model.evaluate(dataset, verbose=0)
    print(f"{model_name} | test loss: {test_loss:.4f} | test accuracy: {test_acc:.4f}")

    probabilities = model.predict(dataset, verbose=0)
    y_pred = np.argmax(probabilities, axis=1)

    if y_true is None:
        y_true = np.concatenate([y.numpy() for _, y in dataset], axis=0)

    print(classification_report(y_true, y_pred, target_names=class_names, digits=3))
    return y_pred, probabilities, test_acc


## 7. Model 1 — Shallow baseline CNN


In [ ]:
def build_shallow_cnn(use_augmentation=False):
    inputs = keras.Input(shape=(32, 32, 3))

    x = layers.Rescaling(1./255)(inputs)

    if use_augmentation:
        x = augmentation(x)

    x = normalizer(x)

    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.30)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    return keras.Model(inputs, outputs, name="Shallow_CNN")


In [ ]:
model_1 = build_shallow_cnn(use_augmentation=False)

history_1 = train_model(
    model_1,
    "model_1_shallow_cnn",
    patience=6
)

plot_history(
    history_1,
    "Model 1 Shallow CNN"
)

preds_1, probs_1, acc_1 = evaluate_model(
    model_1,
    test_ds,
    y_true=y_test,
    model_name="Model 1 — Shallow CNN"
)


## 8. Model 1b — Shallow CNN with data augmentation


In [ ]:
model_1_aug = build_shallow_cnn(use_augmentation=True)

history_1_aug = train_model(
    model_1_aug,
    "model_1b_shallow_cnn_aug",
    patience=6
)

plot_history(
    history_1_aug,
    "Model 1b Shallow CNN Aug"
)

preds_1_aug, probs_1_aug, acc_1_aug = evaluate_model(
    model_1_aug,
    test_ds,
    y_true=y_test,
    model_name="Model 1b — Shallow CNN + Augmentation"
)


## 9. Model 2 — Deep CNN


In [ ]:
def conv_block(x, filters, pool=True, weight_decay=1e-4):
    x = layers.Conv2D(
        filters,
        3,
        padding="same",
        use_bias=False,
        kernel_regularizer=keras.regularizers.l2(weight_decay)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv2D(
        filters,
        3,
        padding="same",
        use_bias=False,
        kernel_regularizer=keras.regularizers.l2(weight_decay)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    if pool:
        x = layers.MaxPooling2D()(x)

    return x


def build_deep_cnn(use_augmentation=True):
    inputs = keras.Input(shape=(32, 32, 3))

    x = layers.Rescaling(1./255)(inputs)

    if use_augmentation:
        x = augmentation(x)

    x = normalizer(x)

    x = conv_block(x, 64, pool=True)
    x = conv_block(x, 128, pool=True)
    x = conv_block(x, 256, pool=True)
    x = conv_block(x, 256, pool=False)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.50)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    return keras.Model(inputs, outputs, name="Deep_CNN_BN_Dropout")


In [ ]:
model_2 = build_deep_cnn(use_augmentation=True)

history_2 = train_model(
    model_2,
    "model_2_deep_cnn",
    patience=7
)

plot_history(
    history_2,
    "Model 2 Deep CNN"
)

preds_2, probs_2, acc_2 = evaluate_model(
    model_2,
    test_ds,
    y_true=y_test,
    model_name="Model 2 — Deep CNN"
)


## 10. Model 2 experiments — Learning rate, clipping and regularization


In [ ]:
model_2_lr_5e4 = build_deep_cnn(use_augmentation=True)

history_2_lr_5e4 = train_model(
    model_2_lr_5e4,
    "model_2_deep_cnn_lr_5e4",
    learning_rate=5e-4,
    patience=7
)

plot_history(
    history_2_lr_5e4,
    "Model 2 Deep CNN — Learning rate 5e-4"
)

preds_2_lr_5e4, probs_2_lr_5e4, acc_2_lr_5e4 = evaluate_model(
    model_2_lr_5e4,
    test_ds,
    y_true=y_test,
    model_name="Model 2 — Deep CNN lr=5e-4"
)

print("Test accuracy with lr=5e-4:", acc_2_lr_5e4)


In [ ]:
model_2_clip = build_deep_cnn(use_augmentation=True)

history_2_clip = train_model_clip(
    model_2_clip,
    "model_2_deep_cnn_lr_5e4_clipnorm",
    learning_rate=5e-4,
    patience=7
)

plot_history(
    history_2_clip,
    "Model 2 Deep CNN — lr=5e-4 + clipnorm"
)

preds_2_clip, probs_2_clip, acc_2_clip = evaluate_model(
    model_2_clip,
    test_ds,
    y_true=y_test,
    model_name="Model 2 — Deep CNN lr=5e-4 + clipnorm"
)

print("Test accuracy Deep CNN lr=5e-4 + clipnorm:", acc_2_clip)


In [ ]:
def build_deep_cnn_l2_2e4(use_augmentation=True):
    inputs = keras.Input(shape=(32, 32, 3))

    x = layers.Rescaling(1./255)(inputs)

    if use_augmentation:
        x = augmentation(x)

    x = normalizer(x)

    x = conv_block(x, 64, pool=True, weight_decay=2e-4)
    x = conv_block(x, 128, pool=True, weight_decay=2e-4)
    x = conv_block(x, 256, pool=True, weight_decay=2e-4)
    x = conv_block(x, 256, pool=False, weight_decay=2e-4)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(
        512,
        use_bias=False,
        kernel_regularizer=keras.regularizers.l2(2e-4)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.50)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    return keras.Model(inputs, outputs, name="Deep_CNN_L2_2e4")


model_2_l2_2e4 = build_deep_cnn_l2_2e4(use_augmentation=True)

history_2_l2_2e4 = train_model_clip(
    model_2_l2_2e4,
    "model_2_deep_cnn_lr_5e4_clipnorm_l2_2e4",
    learning_rate=5e-4,
    patience=7
)

plot_history(
    history_2_l2_2e4,
    "Model 2 Deep CNN — lr=5e-4 + clipnorm + L2 2e-4"
)

preds_2_l2_2e4, probs_2_l2_2e4, acc_2_l2_2e4 = evaluate_model(
    model_2_l2_2e4,
    test_ds,
    y_true=y_test,
    model_name="Model 2 — Deep CNN lr=5e-4 + clipnorm + L2 2e-4"
)

print("Test accuracy Deep CNN lr=5e-4 + clipnorm + L2 2e-4:", acc_2_l2_2e4)


In [ ]:
def build_deep_cnn_dense_256(use_augmentation=True):
    inputs = keras.Input(shape=(32, 32, 3))

    x = layers.Rescaling(1./255)(inputs)

    if use_augmentation:
        x = augmentation(x)

    x = normalizer(x)

    x = conv_block(x, 64, pool=True, weight_decay=1e-4)
    x = conv_block(x, 128, pool=True, weight_decay=1e-4)
    x = conv_block(x, 256, pool=True, weight_decay=1e-4)
    x = conv_block(x, 256, pool=False, weight_decay=1e-4)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.50)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    return keras.Model(inputs, outputs, name="Deep_CNN_Dense_256")


model_2_dense_256 = build_deep_cnn_dense_256(use_augmentation=True)

history_2_dense_256 = train_model_clip(
    model_2_dense_256,
    "model_2_deep_cnn_lr_5e4_clipnorm_dense_256",
    learning_rate=5e-4,
    patience=7
)

plot_history(
    history_2_dense_256,
    "Model 2 Deep CNN — lr=5e-4 + clipnorm + Dense 256"
)

preds_2_dense_256, probs_2_dense_256, acc_2_dense_256 = evaluate_model(
    model_2_dense_256,
    test_ds,
    y_true=y_test,
    model_name="Model 2 — Deep CNN lr=5e-4 + clipnorm + Dense 256"
)

print("Test accuracy Deep CNN lr=5e-4 + clipnorm + Dense 256:", acc_2_dense_256)


## 11. Model 3 — Residual CNN


In [ ]:
def residual_block(x, filters, stride=1, weight_decay=1e-4):
    shortcut = x

    x = layers.Conv2D(
        filters,
        3,
        strides=stride,
        padding="same",
        use_bias=False,
        kernel_regularizer=keras.regularizers.l2(weight_decay)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv2D(
        filters,
        3,
        padding="same",
        use_bias=False,
        kernel_regularizer=keras.regularizers.l2(weight_decay)
    )(x)
    x = layers.BatchNormalization()(x)

    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(
            filters,
            1,
            strides=stride,
            padding="same",
            use_bias=False
        )(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.Activation("relu")(x)

    return x


def build_res_cnn(use_augmentation=True):
    inputs = keras.Input(shape=(32, 32, 3))

    x = layers.Rescaling(1./255)(inputs)

    if use_augmentation:
        x = augmentation(x)

    x = normalizer(x)

    x = layers.Conv2D(64, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = residual_block(x, 64)
    x = residual_block(x, 64)

    x = residual_block(x, 128, stride=2)
    x = residual_block(x, 128)

    x = residual_block(x, 256, stride=2)   
    x = residual_block(x, 256)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.40)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    return keras.Model(inputs, outputs, name="Residual_CNN")


In [ ]:
model_3 = build_res_cnn(use_augmentation=True)

history_3 = train_model(
    model_3,
    "model_3_res_cnn",
    patience=7
)

plot_history(
    history_3,
    "Model 3 Residual CNN"
)

preds_3, probs_3, acc_3 = evaluate_model(
    model_3,
    test_ds,
    y_true=y_test,
    model_name="Model 3 — Residual CNN"
)


## 12. Model 3 experiments — Clipping and stronger augmentation


In [ ]:
model_3_clip = build_res_cnn(use_augmentation=True)

history_3_clip = train_model_clip(
    model_3_clip,
    "model_3_res_cnn_lr_5e4_clipnorm",
    learning_rate=5e-4,
    patience=7
)

plot_history(
    history_3_clip,
    "Model 3 Residual CNN — lr=5e-4 + clipnorm"
)

preds_3_clip, probs_3_clip, acc_3_clip = evaluate_model(
    model_3_clip,
    test_ds,
    y_true=y_test,
    model_name="Model 3 — Residual CNN lr=5e-4 + clipnorm"
)

print("Test accuracy Residual CNN lr=5e-4 + clipnorm:", acc_3_clip)


In [ ]:
augmentation_stronger = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomTranslation(height_factor=0.12, width_factor=0.12, fill_mode="reflect"),
    layers.RandomZoom(height_factor=0.08, width_factor=0.08, fill_mode="reflect"),
    layers.RandomContrast(0.15),
], name="data_augmentation_stronger")


def build_res_cnn_stronger_aug(use_augmentation=True):
    inputs = keras.Input(shape=(32, 32, 3))

    x = layers.Rescaling(1./255)(inputs)

    if use_augmentation:
        x = augmentation_stronger(x)

    x = normalizer(x)

    x = layers.Conv2D(64, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = residual_block(x, 64)
    x = residual_block(x, 64)

    x = residual_block(x, 128, stride=2)
    x = residual_block(x, 128)

    x = residual_block(x, 256, stride=2)
    x = residual_block(x, 256)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.40)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    return keras.Model(inputs, outputs, name="Residual_CNN_Stronger_Aug")


model_3_stronger_aug = build_res_cnn_stronger_aug(use_augmentation=True)

history_3_stronger_aug = train_model_clip(
    model_3_stronger_aug,
    "model_3_res_cnn_lr_5e4_clipnorm_stronger_aug",
    train_ds=train_ds,
    val_ds=val_ds,
    epochs=EPOCHS,
    learning_rate=5e-4,
    patience=7
)

plot_history(
    history_3_stronger_aug,
    "Model 3 Residual CNN — lr=5e-4 + clipnorm + Stronger Augmentation"
)

preds_3_stronger_aug, probs_3_stronger_aug, acc_3_stronger_aug = evaluate_model(
    model_3_stronger_aug,
    test_ds,
    y_true=y_test,
    model_name="Model 3 — Residual CNN lr=5e-4 + clipnorm + Stronger Augmentation"
)

print("Test accuracy Residual CNN stronger augmentation:", acc_3_stronger_aug)


## 13. Transfer learning datasets


In [ ]:
def preprocess_for_mobilenet(x, y, training=False):
    x = tf.cast(x, tf.float32)
    x = tf.image.resize(x, (TL_IMG_SIZE, TL_IMG_SIZE))

    if training:
        x = tf.image.random_flip_left_right(x, seed=SEED)

    x = keras.applications.mobilenet_v2.preprocess_input(x)
    return x, y


def make_tl_dataset(x, y, training=False, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices((x, y))

    if training:
        ds = ds.shuffle(len(x), seed=SEED, reshuffle_each_iteration=True)

    ds = ds.map(
        lambda a, b: preprocess_for_mobilenet(a, b, training=training),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return ds


tl_train_ds = make_tl_dataset(x_train, y_train, training=True)
tl_val_ds = make_tl_dataset(x_val, y_val, training=False)
tl_test_ds = make_tl_dataset(x_test, y_test, training=False)

print(tl_train_ds)


## 14. Model 4 — MobileNetV2 transfer learning


In [ ]:
def build_mobilenet_transfer(freeze_backbone=True):
    base_model = keras.applications.MobileNetV2(
        input_shape=(TL_IMG_SIZE, TL_IMG_SIZE, 3),
        include_top=False,
        weights="imagenet",
        pooling="avg"
    )
    base_model.trainable = not freeze_backbone

    inputs = keras.Input(shape=(TL_IMG_SIZE, TL_IMG_SIZE, 3))

    x = base_model(inputs, training=False)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(0.40)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    return keras.Model(inputs, outputs, name="MobileNetV2_transfer")


In [ ]:
model_4 = build_mobilenet_transfer(freeze_backbone=True)

history_4 = train_model(
    model_4,
    "model_4_mobilenetv2_transfer",
    train_ds=tl_train_ds,
    val_ds=tl_val_ds,
    epochs=EPOCHS,
    learning_rate=1e-3,
    patience=6
)

plot_history(
    history_4,
    "Model 4 MobileNetV2"
)

preds_4, probs_4, acc_4 = evaluate_model(
    model_4,
    tl_test_ds,
    y_true=y_test,
    model_name="Model 4 — MobileNetV2 Transfer"
)


## 15. Model 4 — MobileNetV2 fine-tuning


In [ ]:
base_model = model_4.get_layer(index=1)
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

compile_model(model_4, learning_rate=1e-5)

history_4_ft = model_4.fit(
    tl_train_ds,
    validation_data=tl_val_ds,
    epochs=10,
    callbacks=get_callbacks("model_4_mobilenetv2_finetuned", patience=4),
    verbose=1
)

plot_history(
    history_4_ft,
    "Model 4 MobileNetV2 Fine Tuning"
)

preds_4_ft, probs_4_ft, acc_4_ft = evaluate_model(
    model_4,
    tl_test_ds,
    y_true=y_test,
    model_name="Model 4 — MobileNetV2 Fine-tuned"
)


## 16. Model 4 — MobileNetV2 with stronger augmentation


In [ ]:
def preprocess_for_mobilenet_aug(x, y, training=False):
    x = tf.cast(x, tf.float32)

    if training:
        x = tf.image.resize(x, (TL_IMG_SIZE + 8, TL_IMG_SIZE + 8))
        x = tf.image.random_crop(x, size=(TL_IMG_SIZE, TL_IMG_SIZE, 3), seed=SEED)
        x = tf.image.random_flip_left_right(x, seed=SEED)
        x = tf.image.random_brightness(x, max_delta=0.05)
        x = tf.image.random_contrast(x, lower=0.90, upper=1.10)
    else:
        x = tf.image.resize(x, (TL_IMG_SIZE, TL_IMG_SIZE))

    x = keras.applications.mobilenet_v2.preprocess_input(x)

    return x, y


def make_tl_dataset_aug(x, y, training=False, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices((x, y))

    if training:
        ds = ds.shuffle(len(x), seed=SEED, reshuffle_each_iteration=True)

    ds = ds.map(
        lambda a, b: preprocess_for_mobilenet_aug(a, b, training=training),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return ds


tl_train_ds_aug = make_tl_dataset_aug(x_train, y_train, training=True)
tl_val_ds_aug = make_tl_dataset_aug(x_val, y_val, training=False)
tl_test_ds_aug = make_tl_dataset_aug(x_test, y_test, training=False)


model_4_aug = build_mobilenet_transfer(freeze_backbone=True)

history_4_aug = train_model(
    model_4_aug,
    "model_4_mobilenetv2_same_head_aug",
    train_ds=tl_train_ds_aug,
    val_ds=tl_val_ds_aug,
    epochs=EPOCHS,
    learning_rate=1e-3,
    patience=6
)

plot_history(
    history_4_aug,
    "Model 4 MobileNetV2 — Same Head + Stronger Augmentation"
)

preds_4_aug, probs_4_aug, acc_4_aug = evaluate_model(
    model_4_aug,
    tl_test_ds_aug,
    y_true=y_test,
    model_name="Model 4 — MobileNetV2 Same Head + Stronger Augmentation"
)

print("Test accuracy MobileNetV2 same head + stronger augmentation:", acc_4_aug)


## 17. Model 4 — MobileNetV2 stronger augmentation + fine-tuning


In [ ]:
base_model = model_4_aug.layers[1]
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model_4_aug.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-5,
        clipnorm=1.0
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_4_aug_ft = model_4_aug.fit(
    tl_train_ds_aug,
    validation_data=tl_val_ds_aug,
    epochs=20,
    callbacks=get_callbacks("model_4_mobilenetv2_aug_finetuned", patience=4),
    verbose=1
)

plot_history(
    history_4_aug_ft,
    "Model 4 MobileNetV2 — Stronger Augmentation + Fine Tuning"
)

preds_4_aug_ft, probs_4_aug_ft, acc_4_aug_ft = evaluate_model(
    model_4_aug,
    tl_test_ds_aug,
    y_true=y_test,
    model_name="Model 4 — MobileNetV2 Stronger Augmentation + Fine-tuned"
)

print("Test accuracy MobileNetV2 stronger augmentation + fine-tuned:", acc_4_aug_ft)


## 18. Model 5 — ResNet50 transfer learning


In [ ]:
def preprocess_for_resnet50(x, y, training=False):
    x = tf.cast(x, tf.float32)
    x = tf.image.resize(x, (TL_IMG_SIZE, TL_IMG_SIZE))

    if training:
        x = tf.image.random_flip_left_right(x, seed=SEED)

    x = keras.applications.resnet50.preprocess_input(x)

    return x, y


def make_resnet50_dataset(x, y, training=False, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices((x, y))

    if training:
        ds = ds.shuffle(len(x), seed=SEED, reshuffle_each_iteration=True)

    ds = ds.map(
        lambda a, b: preprocess_for_resnet50(a, b, training=training),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return ds


def build_resnet50_transfer(freeze_backbone=True):
    base_model = keras.applications.ResNet50(
        input_shape=(TL_IMG_SIZE, TL_IMG_SIZE, 3),
        include_top=False,
        weights="imagenet",
        pooling="avg"
    )
    base_model.trainable = not freeze_backbone

    inputs = keras.Input(shape=(TL_IMG_SIZE, TL_IMG_SIZE, 3))

    x = base_model(inputs, training=False)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(0.50)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    return keras.Model(inputs, outputs, name="ResNet50_transfer")


resnet_train_ds = make_resnet50_dataset(x_train, y_train, training=True)
resnet_val_ds = make_resnet50_dataset(x_val, y_val, training=False)
resnet_test_ds = make_resnet50_dataset(x_test, y_test, training=False)

model_5 = build_resnet50_transfer(freeze_backbone=True)

history_5 = train_model(
    model_5,
    "model_5_resnet50_transfer",
    train_ds=resnet_train_ds,
    val_ds=resnet_val_ds,
    epochs=EPOCHS,
    learning_rate=5e-4,
    patience=6
)

plot_history(
    history_5,
    "Model 5 ResNet50"
)

preds_5, probs_5, acc_5 = evaluate_model(
    model_5,
    resnet_test_ds,
    y_true=y_test,
    model_name="Model 5 — ResNet50 Transfer"
)


## 19. Final comparison


In [ ]:
result_variables = [
    ("Shallow CNN", "acc_1", "preds_1"),
    ("Shallow CNN + Aug", "acc_1_aug", "preds_1_aug"),
    ("Deep CNN", "acc_2", "preds_2"),
    ("Deep CNN lr=5e-4", "acc_2_lr_5e4", "preds_2_lr_5e4"),
    ("Deep CNN lr=5e-4 + clipnorm", "acc_2_clip", "preds_2_clip"),
    ("Deep CNN L2 2e-4", "acc_2_l2_2e4", "preds_2_l2_2e4"),
    ("Deep CNN Dense 256", "acc_2_dense_256", "preds_2_dense_256"),
    ("Residual CNN", "acc_3", "preds_3"),
    ("Residual CNN lr=5e-4 + clipnorm", "acc_3_clip", "preds_3_clip"),
    ("Residual CNN stronger aug", "acc_3_stronger_aug", "preds_3_stronger_aug"),
    ("MobileNetV2 transfer", "acc_4", "preds_4"),
    ("MobileNetV2 fine-tuned", "acc_4_ft", "preds_4_ft"),
    ("MobileNetV2 stronger aug", "acc_4_aug", "preds_4_aug"),
    ("MobileNetV2 stronger aug + FT", "acc_4_aug_ft", "preds_4_aug_ft"),
    ("ResNet50 transfer", "acc_5", "preds_5"),
]

results = {
    name: globals()[acc_var]
    for name, acc_var, _ in result_variables
    if acc_var in globals()
}

predictions_by_model = {
    name: globals()[pred_var]
    for name, _, pred_var in result_variables
    if pred_var in globals()
}

plt.figure(figsize=(12, 6))
names = list(results.keys())
values = [results[k] * 100 for k in names]

bars = plt.bar(names, values)
plt.ylabel("Test accuracy (%)")
plt.ylim(0, 100)
plt.title("Model comparison on CIFAR-10", fontweight="bold")
plt.xticks(rotation=35, ha="right")
plt.grid(axis="y", alpha=0.3)

for bar, value in zip(bars, values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{value:.1f}%",
        ha="center",
        fontsize=8
    )

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "model_comparison.png"), dpi=120, bbox_inches="tight")
plt.show()

best_model_name = max(results, key=results.get)
best_test_accuracy = results[best_model_name]
best_preds = predictions_by_model[best_model_name]

print(f"Best model: {best_model_name}")
print(f"Best test accuracy: {best_test_accuracy:.4f}")


## 20. Confusion matrix and error analysis


In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names, title="Confusion matrix"):
    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(10, 10))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)

    ax.set_title(title, fontweight="bold")
    plt.tight_layout()

    filename = title.replace(" ", "_").replace("—", "-").lower() + ".png"
    plt.savefig(os.path.join(OUT_DIR, filename), dpi=120, bbox_inches="tight")
    plt.show()


plot_confusion_matrix(
    y_test,
    best_preds,
    class_names,
    title=f"Confusion matrix — {best_model_name}"
)


In [ ]:
def show_misclassified_images(x, y_true, y_pred, class_names, n=36):
    wrong = np.where(y_true != y_pred)[0]
    selected = wrong[:n]
    rows = int(np.ceil(n / 6))

    plt.figure(figsize=(12, 2 * rows))
    for i, idx in enumerate(selected):
        ax = plt.subplot(rows, 6, i + 1)
        ax.imshow(x[idx].astype("uint8"))
        ax.set_title(
            f"true: {class_names[y_true[idx]]}\npred: {class_names[y_pred[idx]]}",
            fontsize=8
        )
        ax.axis("off")

    plt.suptitle("Misclassified images", fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "misclassified_images.png"), dpi=120, bbox_inches="tight")
    plt.show()


show_misclassified_images(
    x_test,
    y_test,
    best_preds,
    class_names
)


## 21. Saving final models


In [ ]:
models_to_save = {
    "model_1": "model_1_shallow_cnn.keras",
    "model_1_aug": "model_1b_shallow_cnn_aug.keras",
    "model_2": "model_2_deep_cnn.keras",
    "model_2_clip": "model_2_deep_cnn_lr_5e4_clipnorm.keras",
    "model_3": "model_3_residual_cnn.keras",
    "model_3_clip": "model_3_residual_cnn_lr_5e4_clipnorm.keras",
    "model_4": "model_4_mobilenetv2_transfer_or_finetuned.keras",
    "model_4_aug": "model_4_mobilenetv2_stronger_aug_finetuned.keras",
    "model_5": "model_5_resnet50_transfer.keras",
}

for model_var, filename in models_to_save.items():
    if model_var in globals():
        globals()[model_var].save(os.path.join(OUT_DIR, filename))
        print(f"Saved {filename}")

print("Models saved in", OUT_DIR)
